# LLM Inference with Python

Now that Ollama is running on your machine, let's interact with it **programmatically**. This is where LLMs go from a fun toy to a real engineering tool — you can build chatbots, automation pipelines, data extractors, and more.

The `ollama` Python library talks to the Ollama server running on `localhost:11434` via HTTP. Every call you make goes through this flow:

```
Your Python Code → ollama library → HTTP request → Ollama Server → Model → Tokens → Response
```

## 1. Install the Library

In [1]:
!pip install ollama -q

## 2. Two APIs: `generate()` vs `chat()`

Ollama gives you two distinct ways to talk to a model. Understanding the difference is important:

| Feature | `ollama.generate()` | `ollama.chat()` |
| :--- | :--- | :--- |
| Input | A single text prompt | A list of messages (with roles) |
| Best for | Single completions, one-off tasks | Conversations, multi-turn chat |
| System prompt | Via `system` parameter | Via `{'role': 'system'}` message |
| History | None (stateless) | You manage the message list |

Let's try both.

### 2a. `generate()` — Simple One-Shot Completion

In [2]:
import ollama

response = ollama.generate(
    model='llama3',
    prompt='Explain the importance of geospatial data in 3 bullet points.'
)

print(response['response'])

Here are three key reasons why geospatial data is important:

• **Informed Decision Making**: Geospatial data provides critical information to support informed decision making in various fields such as urban planning, emergency response, natural resource management, and economic development. By analyzing geographic patterns and relationships, policymakers, researchers, and practitioners can identify trends, optimize resources, and develop targeted strategies that improve outcomes.

• **Improved Resource Allocation**: Geospatial data helps allocate resources efficiently by identifying areas of need or opportunity. For instance, in emergency response, geospatial data can pinpoint the location of affected populations, infrastructure damage, and resource availability to prioritize relief efforts. Similarly, in business, geospatial analysis can help companies optimize supply chain management, logistics, and marketing efforts.

• **Enhanced Understanding of Complex Systems**: Geospatial data

### 2b. `chat()` — Role-Based Messages

The `chat()` API uses a messages format with **roles**:
- `system`: Sets the model's behavior/persona (optional but powerful)
- `user`: Your message
- `assistant`: The model's previous responses (for maintaining context)

In [3]:
response = ollama.chat(model='llama3', messages=[
  {
    'role': 'GIS Analyst',
    'content': 'Explain the importance of geospatial data in 3 bullet points.',
  },
])

print(response['message']['content'])

I'd be happy to help!


## 3. Streaming Responses

When the model generates a long response, waiting for the *entire* thing to finish before seeing anything is frustrating. **Streaming** lets you see tokens as they're generated — just like ChatGPT.

Under the hood, streaming uses Server-Sent Events (SSE) — the server sends tiny chunks of the response as they're generated instead of buffering everything.

In [4]:
stream = ollama.chat(
    model='llama3',
    messages=[{'role': 'user', 'content': 'Write a short story about a rabbit exploring a desert.'}],
    stream=True,
)

for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

As the sun rose over the vast expanse of sand, a curious rabbit named Rosie hopped out of her burrow and set off to explore the desert. She had never ventured beyond her familiar surroundings before, but something about the warm breeze and the endless dunes called to her.

Rosie's long ears twitched with excitement as she traversed the soft sand, her paws sinking into the fine grains. The air was filled with the sweet scent of creosote bushes and the distant chirping of birds. She sniffed and snuffled, taking in every new smell and sound.

As she wandered, Rosie discovered a small oasis hidden behind a wall of rock. A cluster of palm trees surrounded a tiny spring, which gurgled and splashed into a shallow pool. The rabbit's eyes shone with delight as she approached the water's edge. She lapped up the cool liquid, feeling it quench her thirst on this hot desert day.

Next to the oasis, Rosie found a scattering of strange, spindly plants. Their leaves were covered in fine, white hairs t

## 4. Controlling the Output: `options` Parameter

Remember temperature and top_p from the basics notebook? Here's how you actually use them in code. The `options` dict lets you control many aspects of generation:

In [5]:
# Factual mode: low temperature, focused output
response = ollama.generate(
    model='llama3',
    prompt='What is NDVI and why is it used in agriculture?',
    options={
        'temperature': 0.2,   # Very deterministic
        'top_p': 0.9,         # Consider top 90% probability mass
        'num_predict': 150,   # Max tokens to generate
    }
)

print("📊 Factual mode (temp=0.2):")
print(response['response'])

📊 Factual mode (temp=0.2):
NDVI stands for Normalized Difference Vegetation Index. It's a widely used remote sensing index that measures the health and density of vegetation, such as crops, forests, or grasslands. In agriculture, NDVI is employed to monitor crop growth, detect stress, and optimize farming practices.

Here's how it works:

1. **Satellite imagery**: Satellites like NASA's Landsat or Sentinel-2 capture images of the Earth's surface.
2. **Vegetation indices**: The images are analyzed using various vegetation indices, such as NDVI, to quantify the amount of green vegetation present.
3. **NDVI calculation**: The index is calculated by subtracting the near-infrared (NIR) reflectance from the red reflectance


In [6]:
# Creative mode: high temperature, diverse output
response = ollama.generate(
    model='llama3',
    prompt='Write a haiku about satellite imagery.',
    options={
        'temperature': 1.0,   # Very creative
        'top_p': 0.95,
        'num_predict': 50,
    }
)

print("🎨 Creative mode (temp=1.0):")
print(response['response'])

🎨 Creative mode (temp=1.0):
Here is a haiku about satellite imagery:

Clouds of data pour
Earthly secrets revealed high
Aerial gaze deep


### Full Options Reference

| Option | Default | What it Does |
| :--- | :--- | :--- |
| `temperature` | 0.8 | Controls randomness (0 = deterministic, 2 = chaos) |
| `top_p` | 0.9 | Nucleus sampling — limits token pool |
| `top_k` | 40 | Only sample from top K tokens |
| `num_predict` | 128 | Maximum tokens to generate |
| `repeat_penalty` | 1.1 | Penalizes repeating tokens (higher = less repetition) |
| `seed` | random | Set a fixed seed for reproducible outputs |

## 5. Structured Output (JSON)

One of the most practical LLM skills: forcing the model to return **structured data** that your code can parse. This is huge for automation — instead of getting a paragraph you need to regex through, you get clean JSON.

**How it works:** The `format='json'` parameter constrains the model's token generation so it can *only* produce valid JSON tokens. It's not 100% reliable with complex schemas, but for simple extractions it works great.

In [7]:
import json

response = ollama.chat(
    model='llama3',
    messages=[{'role': 'user', 'content': 'Extract the location and emotion from: "I am so happy to be in Sargodha today!"'}],
    format='json',
)

# Parse the JSON response
raw = response['message']['content']
data = json.loads(raw)

print(f"Raw JSON: {raw}")
print(f"\nParsed data:")
for key, value in data.items():
    print(f"  {key}: {value}")

Raw JSON: {}

Parsed data:


In [8]:
# More complex extraction — multiple entities from a geospatial description
geo_text = """The 7.8 magnitude earthquake struck southeastern Turkey near Gaziantep 
on February 6, 2023. Satellite imagery from Sentinel-2 showed widespread 
building damage across an area of approximately 350 sq km."""

response = ollama.chat(
    model='llama3',
    messages=[
        {'role': 'system', 'content': 'Extract structured data as JSON with keys: event_type, magnitude, location, date, satellite, affected_area'},
        {'role': 'user', 'content': geo_text}
    ],
    format='json',
)

data = json.loads(response['message']['content'])
print(json.dumps(data, indent=2))

{
  "event_type": "earthquake",
  "magnitude": "7.8",
  "location": "southeastern Turkey near Gaziantep",
  "date": "February 6, 2023",
  "satellite": "Sentinel-2",
  "affected_area": "approximately 350 sq km"
}


## 6. Multi-Turn Conversation (Chat with Memory)

This is where `chat()` really shines. LLMs are stateless by default — each API call is independent. To create a "conversation", **you** maintain the message history and send the full conversation each time.

Think of it like showing someone your entire chat log before each reply, because the model has no memory between calls.

In [9]:
# Multi-turn conversation example
messages = [
    {'role': 'system', 'content': 'You are a helpful geospatial data expert. Keep answers concise.'},
]

def chat(user_msg):
    """Send a message and maintain conversation history."""
    messages.append({'role': 'user', 'content': user_msg})
    
    response = ollama.chat(model='llama3', messages=messages)
    assistant_msg = response['message']['content']
    
    messages.append({'role': 'assistant', 'content': assistant_msg})
    return assistant_msg

# Turn 1
print("👤 User: What is NDVI?")
print(f"🤖 AI: {chat('What is NDVI?')}")
print()

# Turn 2 — the model remembers the context!
print("👤 User: What range of values does it typically have?")
print(f"🤖 AI: {chat('What range of values does it typically have?')}")
print()

# Turn 3 — still maintains context
print("👤 User: Give me the Python formula.")
print(f"🤖 AI: {chat('Give me the Python formula.')}")

👤 User: What is NDVI?
🤖 AI: NDVI (Normalized Difference Vegetation Index) is a satellite-based vegetation health indicator, measuring the difference between reflected red and infrared light. It's calculated as: (NIR - Red) / (NIR + Red). Higher values indicate healthy vegetation, while lower values suggest lack of vegetation or water.

👤 User: What range of values does it typically have?
🤖 AI: NDVI values typically range from -1 to 1. Most natural environments like forests, grasslands, and crops typically have NDVI values between 0.2 and 0.8, with higher values indicating denser vegetation. Water bodies tend to have negative or low NDVI values (-1 to 0.2), while urban areas may have lower or zero NDVI values.

👤 User: Give me the Python formula.
🤖 AI: Here is the Python formula for calculating NDVI:

```
ndvi = (nir - red) / (nir + red)
```

Where `nir` and `red` are the reflectance values in the Near-Infrared (NIR) and Red bands, respectively.


In [10]:
# Let's peek at the conversation history the model receives
print(f"Total messages in history: {len(messages)}")
print(f"Total conversation (approx words): {sum(len(m['content'].split()) for m in messages)}")
print()
for i, msg in enumerate(messages):
    preview = msg['content'][:80] + ('...' if len(msg['content']) > 80 else '')
    print(f"  [{i}] {msg['role']:>10}: {preview}")

Total messages in history: 7
Total conversation (approx words): 158

  [0]     system: You are a helpful geospatial data expert. Keep answers concise.
  [1]       user: What is NDVI?
  [2]  assistant: NDVI (Normalized Difference Vegetation Index) is a satellite-based vegetation he...
  [3]       user: What range of values does it typically have?
  [4]  assistant: NDVI values typically range from -1 to 1. Most natural environments like forests...
  [5]       user: Give me the Python formula.
  [6]  assistant: Here is the Python formula for calculating NDVI:

```
ndvi = (nir - red) / (nir ...


**Key insight:** Every new message includes the *entire* conversation history. As the conversation gets longer, you use more of the context window. Eventually, you'll hit the limit and need strategies like:
- Summarizing old messages
- Dropping early turns
- Using RAG to retrieve relevant context

## 7. Error Handling

In real applications, things go wrong — the server might be down, the model might not exist, or the response might not parse. Here's how to handle it gracefully:

In [11]:
def safe_generate(prompt, model='llama3', **kwargs):
    """A production-ready wrapper around ollama.generate with error handling."""
    try:
        response = ollama.generate(model=model, prompt=prompt, **kwargs)
        return response['response']
    except ConnectionError:
        return "❌ Error: Ollama server is not running. Start it with 'ollama serve'."
    except ollama.ResponseError as e:
        if 'not found' in str(e).lower():
            return f"❌ Error: Model '{model}' not found. Pull it with: ollama pull {model}"
        return f"❌ Error: {e}"
    except Exception as e:
        return f"❌ Unexpected error: {type(e).__name__}: {e}"

# Test with a real model
print(safe_generate("Say hello in one word."))

# Test with a nonexistent model
print(safe_generate("Hello", model='nonexistent-model-xyz'))

Hello!
❌ Error: Model 'nonexistent-model-xyz' not found. Pull it with: ollama pull nonexistent-model-xyz


## 8. Practical Example: Batch Processing with LLMs

Let's do something more realistic — process multiple data points with an LLM. This is a common pattern in data science: using LLMs as "intelligent data transformers".

In [12]:
import json

# Simulated sensor reports that need classification
reports = [
    "Water level rising rapidly at Station 14, exceeded 3m threshold.",
    "Air quality normal. PM2.5 at 12 µg/m³, well within safe limits.",
    "Seismic activity detected: 2.1 magnitude tremor near sensor grid B7.",
    "Solar panel output dropped 40% — possible cloud cover or dust accumulation.",
]

print(f"Processing {len(reports)} sensor reports...\n")

for i, report in enumerate(reports, 1):
    response = ollama.chat(
        model='llama3',
        messages=[
            {'role': 'system', 'content': 'Classify the sensor report. Return JSON with: severity (low/medium/high), category (flood/air/seismic/solar/other), action_needed (yes/no).'},
            {'role': 'user', 'content': report}
        ],
        format='json',
    )
    
    result = json.loads(response['message']['content'])
    print(f"Report {i}: {report[:60]}...")
    print(f"  → {result}")
    print()

Processing 4 sensor reports...

Report 1: Water level rising rapidly at Station 14, exceeded 3m thresh...
  → {'severity': 'high', 'category': 'flood', 'action_needed': 'yes'}

Report 2: Air quality normal. PM2.5 at 12 µg/m³, well within safe limi...
  → {'severity': 'low', 'category': 'air', 'action_needed': 'no'}

Report 3: Seismic activity detected: 2.1 magnitude tremor near sensor ...
  → {'severity': 'high', 'category': 'seismic', 'action_needed': 'yes'}

Report 4: Solar panel output dropped 40% — possible cloud cover or dus...
  → {'severity': 'medium', 'category': 'solar', 'action_needed': 'yes'}



## Challenge! 🚀

1. Modify the batch processor above to also extract a `recommended_response` field in the JSON
2. Build a simple interactive chatbot using the multi-turn pattern from Section 6. Use `input()` to get user messages and print the AI response in a loop.

---

**Next notebook:** [Prompt Engineering Guide](./prompt_engineering_guide.ipynb) — Learn how to get *much* better outputs by crafting better prompts.